# Download ground truths

author: laquitainesteeve@gmail.com

purpose: download ground truths from dandi archive

Execution time: 3 secs

Special hardware: on CPU, does not require GPU.



# Setup 

1. Activate dandi virtual environment (envs/dandi.yml)
2. python -m ipykernel install --user --name dandi --display-name "dandi"

In [4]:
%%time 
%load_ext autoreload
%autoreload 2

# import python packages
import os
import numpy as np
from time import time
from dandi.dandiapi import DandiAPIClient
import spikeinterface.extractors as se
import spikeinterface.sorters as ss
import spikeinterface
from pynwb.file import NWBFile, Subject
from pynwb import NWBHDF5IO
import uuid
from datetime import datetime
from dateutil.tz import tzlocal
import spikeinterface as si
import os
import shutil 
import pandas as pd

print("spikeinterface", spikeinterface.__version__)

proj_path = "/home/steeve/steeve/epfl/code/spikebias"
os.chdir(proj_path)

# custom package
from src.nodes.dataloader.dataloader import SortingLoader

# setup parameters
SAVE = True # save sorting extractor

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
spikeinterface 0.101.2
CPU times: user 0 ns, sys: 306 μs, total: 306 μs
Wall time: 248 μs


## Functions

In [5]:
def save_ground_truth_metadata(
    property_data_path: str, Sorting, ground_truth_path: str, save: bool
):
    """add cells properties to Ground truth extractor

    Args:
        property_data_path: path of cell properties h5 file (e.g., assets/metadata/silico_neuropixels/npx_evoked/cell_properties.h5)
        Sorting: Ground truth Sorting Extractor
        ground_truth_path (None): ground truth saving path
        save (bool): whether to save the Sorting Extractor

    Returns:
        Sorting Extractor
    """
    # cell properties to h5 file in the local project path
    df = pd.read_hdf(property_data_path, key="cell_properties")

    # add as property to Sorting Extractor
    for prop in df.columns:
        Sorting.set_property(prop, df[prop].values.tolist())

    # make a "layers" copy of "layer"
    # for convenience
    Sorting.set_property("layers", df["layer"].values.tolist())

    # save Sorting Extractor
    if save:
        shutil.rmtree(ground_truth_path, ignore_errors=True)
        Sorting.save(folder=ground_truth_path)
    return Sorting

## Load ground truth

### Single-cell isolated traces with Reyes probe 

In [3]:
%%time

# load dandiset (npx, spontaneous, 40Khz)
dandiset_id = '001250'
filepath = 'sub-biophy-isolated-traces-reyes/sub-biophy-isolated-traces-reyes_ses-006_ecephys.nwb' # ground truth spikes and unit metadata of biophy npx spontaneous
SFREQ = 20000 # sampling frequency
TSTART = 0    # default - these are timestamps

# Instantiate and load sorting
loader = SortingLoader(dandiset_id, filepath, SFREQ, TSTART)
GroundTruth = loader.load_sorting()

# write
if SAVE:
    GroundTruth.save(folder = os.path.join(proj_path, 'dataset/00_raw/ground_truth_reyes_isolated_traces'))

# report
print('\n', GroundTruth)
GroundTruth


 NwbSortingExtractor: 1 units - 1 segments - 20.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/2f3/edc/2f3edc58-b09e-4571-bb1c-1ffb2f768b57
CPU times: user 114 ms, sys: 1.6 ms, total: 115 ms
Wall time: 9.17 s


NwbSortingExtractor: 1 units - 1 segments - 20.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/2f3/edc/2f3edc58-b09e-4571-bb1c-1ffb2f768b57

### npx_spont

* note: Sorting extractor already contains cell metadata
* Execution time: 3 min

In [3]:
%%time

# load dandiset (npx, spontaneous, 40Khz)
dandiset_id = '001250'
filepath = 'sub-001/sub-001_ecephys.nwb' # ground truth spikes and unit metadata of biophy npx spontaneous
SFREQ = 40000 # sampling frequency
TSTART = 0    # default - these are timestamps

# Instantiate and load sorting
loader = SortingLoader(dandiset_id, filepath, SFREQ, TSTART)
GroundTruth = loader.load_sorting()

# write
if SAVE:
    GroundTruth.save(folder = os.path.join(proj_path, "dataset/00_raw/ground_truth_npx_spont"))

# report
print('\n', GroundTruth)
GroundTruth


 NwbSortingExtractor: 1388 units - 1 segments - 40.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/c96/cd3/c96cd375-a49a-4964-a84b-17f761873005
CPU times: user 610 ms, sys: 163 ms, total: 773 ms
Wall time: 2min 42s


NwbSortingExtractor: 1388 units - 1 segments - 40.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/c96/cd3/c96cd375-a49a-4964-a84b-17f761873005

### npx_evoked

* We add metadata that were not saved in the dandi dataset
* Execution time: 6 secs

In [ ]:
%%time

# load dandiset
DANDISET_ID = '001250'
FILEPATH = 'sub-002-fitted/sub-002-fitted_ecephys.nwb' # ground truth spikes and unit metadata
CELL_PROPERTIES_PATH = "assets/metadata/silico_neuropixels/npx_evoked/cell_properties.h5"
SAVE_PATH = "dataset/00_raw/ground_truth_npx_evoked"
SFREQ = 20000 # sampling frequency
TSTART = 0    # default - these are timestamps

# load ground truth cells and timestamps
loader = SortingLoader(DANDISET_ID, FILEPATH, SFREQ, TSTART)
GroundTruth = loader.load_sorting()

# add cells metadata
GroundTruth = save_ground_truth_metadata(CELL_PROPERTIES_PATH, GroundTruth, SAVE_PATH, save=False)

# write
if SAVE:
    GroundTruth.save(folder = os.path.join(proj_path, "dataset/00_raw/ground_truth_npx_evoked"), overwrite=True)

# report
print('\n', GroundTruth)
GroundTruth


 NwbSortingExtractor: 1836 units - 1 segments - 20.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/9d6/6ed/9d66ed40-af31-43aa-b4ba-246d2206dcad
CPU times: user 834 ms, sys: 137 ms, total: 971 ms
Wall time: 6.67 s


NwbSortingExtractor: 1836 units - 1 segments - 20.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/9d6/6ed/9d66ed40-af31-43aa-b4ba-246d2206dcad

### Dense depth 1


* execution time: 3 secs

In [ ]:
%%time

# load dandiset
DANDISET_ID = '001250'
FILEPATH = 'sub-003-fitted/sub-003-fitted_ecephys.nwb'
CELL_PROPERTIES_PATH = "assets/metadata/dense_spont/probe_1/cell_properties.h5"
SFREQ = 20000
TSTART = 0

# load ground truth cells and timestamps
loader = SortingLoader(DANDISET_ID, FILEPATH, SFREQ, TSTART)
GroundTruth = loader.load_sorting()

# add cells metadata
GroundTruth = save_ground_truth_metadata(CELL_PROPERTIES_PATH, GroundTruth, SAVE_PATH, save=False)

# write
if SAVE:
    GroundTruth.save(folder = os.path.join(proj_path, "dataset/00_raw/ground_truth_dense_probe1"), overwrite=True)

# report
print('\n', GroundTruth)
GroundTruth


 NwbSortingExtractor: 287 units - 1 segments - 20.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/dec/e65/dece6568-cee4-4ade-80bf-c1166a03fe2a
CPU times: user 206 ms, sys: 30.2 ms, total: 236 ms
Wall time: 7.54 s


NwbSortingExtractor: 287 units - 1 segments - 20.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/dec/e65/dece6568-cee4-4ade-80bf-c1166a03fe2a

### Dense depth 2

* execution time: 3 secs

In [9]:
%%time

# load dandiset
DANDISET_ID = '001250'
FILEPATH = 'sub-004-fitted/sub-004-fitted_ecephys.nwb'
CELL_PROPERTIES_PATH = "assets/metadata/dense_spont/probe_2/cell_properties.h5"
SFREQ = 20000
TSTART = 0

# load ground truth cells and timestamps
loader = SortingLoader(DANDISET_ID, FILEPATH, SFREQ, TSTART)
GroundTruth = loader.load_sorting()

# add cells metadata
GroundTruth = save_ground_truth_metadata(CELL_PROPERTIES_PATH, GroundTruth, SAVE_PATH, save=False)

# write
if SAVE:
    GroundTruth.save(folder = os.path.join(proj_path, "dataset/00_raw/ground_truth_dense_probe2"), overwrite=True)

# report
print('\n', GroundTruth)
GroundTruth


 NwbSortingExtractor: 770 units - 1 segments - 20.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/eef/9e9/eef9e95c-fb5b-46d2-a24c-878d8170b5e0
CPU times: user 267 ms, sys: 34.9 ms, total: 301 ms
Wall time: 3.69 s


NwbSortingExtractor: 770 units - 1 segments - 20.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/eef/9e9/eef9e95c-fb5b-46d2-a24c-878d8170b5e0

### Dense depth 3

* execution time: 1 min

In [10]:
%%time

# load dandiset
DANDISET_ID = '001250'
FILEPATH = 'sub-005-fitted/sub-005-fitted_ecephys.nwb'
CELL_PROPERTIES_PATH = "assets/metadata/dense_spont/probe_3/cell_properties.h5"
SFREQ = 20000
TSTART = 0

# load ground truth cells and timestamps
loader = SortingLoader(DANDISET_ID, FILEPATH, SFREQ, TSTART)
GroundTruth = loader.load_sorting()

# add cells metadata
GroundTruth = save_ground_truth_metadata(CELL_PROPERTIES_PATH, GroundTruth, SAVE_PATH, save=False)

# write
if SAVE:
    GroundTruth.save(folder = os.path.join(proj_path, "dataset/00_raw/ground_truth_dense_probe3"), overwrite=True)

# report
print('\n', GroundTruth)
GroundTruth


 NwbSortingExtractor: 1123 units - 1 segments - 20.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/ee2/816/ee2816de-d861-4b55-9cde-416a52e54049
CPU times: user 371 ms, sys: 37.8 ms, total: 409 ms
Wall time: 49.5 s


NwbSortingExtractor: 1123 units - 1 segments - 20.0kHz
  file_path: https://dandiarchive.s3.amazonaws.com/blobs/ee2/816/ee2816de-d861-4b55-9cde-416a52e54049